[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# *CI/CD*, cadena de suministro y riesgos actuales.

Este cuaderno introduce el contexto que da origen a este curso: los ataques a la infraestructura de entrega continua no son escenarios hipotéticos, son incidentes documentados que afectaron a organizaciones reales con pipelines similares a los que este curso enseña a construir.

## Objetivos.

- Entender por qué el *pipeline* de *CI/CD* es infraestructura crítica y no solo automatización conveniente.
- Identificar ataques reales y documentados a la cadena de suministro de software.
- Comprender la superficie de ataque de un *pipeline* moderno.
- Entender el modelo de confianza que un *pipeline* *DevSecOps* debe verificar explícitamente.
- Introducir *shift left* como estrategia de defensa distribuida en todo el ciclo de entrega.
- Situar las herramientas y prácticas de este curso frente a las amenazas actuales.

## Por qué este curso existe ahora.

Durante años, la seguridad del software se ocupó principalmente del software en ejecución: vulnerabilidades en el código, controles de acceso, cifrado en tránsito y en reposo. El *pipeline* de construcción y entrega era considerado infraestructura interna de confianza implícita.

Esa suposición resultó ser incorrecta.

En los últimos años, atacantes sofisticados han desplazado su foco hacia el proceso de construcción. La lógica es directa: si puedes comprometer el sistema que construye el software, puedes insertar código malicioso antes de que el producto final llegue a producción, firmado y distribuido con la credibilidad del proveedor legítimo.

El impacto potencial de este desplazamiento supera al de un ataque a la aplicación en producción:

- un sistema de producción comprometido afecta a los usuarios de una organización;
- un sistema de *build* comprometido puede afectar a todos los clientes de esa organización.

Esta es la lógica de los ataques a la cadena de suministro de software (*software supply chain attacks*): no atacar el producto final, sino el proceso que lo produce.

*GitHub Actions* y los *pipelines* de *CI/CD* se han convertido en vectores de ataque de primera línea porque concentran acceso privilegiado a secretos de nube, ejecución automática de código externo y capacidad de despliegue a producción en un solo sistema automatizado que corre en respuesta a eventos del repositorio.

## *CI/CD* como infraestructura crítica.

Un *pipeline* de *CI/CD* moderno es, en términos operativos, un sistema privilegiado:

- tiene acceso a las credenciales de nube que crean y destruyen infraestructura;
- puede publicar paquetes e imágenes en registros públicos o privados con la identidad de la organización;
- tiene acceso al código fuente completo;
- ejecuta código arbitrario en respuesta a eventos del repositorio, incluyendo código de terceros no auditado;
- puede interactuar con sistemas de producción para desplegar cambios de forma automática.

Esa combinación —acceso privilegiado, ejecución automática de código externo, superficie amplia de entrada— convierte al *pipeline* en un objetivo de alto valor para atacantes que buscan acceso persistente o capacidad de modificar artefactos distribuidos a gran escala.

Tratar el *pipeline* como infraestructura interna de confianza implícita equivale a proteger la caja fuerte pero dejar la llave en la puerta trasera. Cada decisión técnica en el *pipeline* —qué *Actions* usar, cómo gestionar secretos, cómo construir y publicar artefactos— tiene consecuencias de seguridad concretas y verificables.

## La superficie de ataque de un *pipeline*.

Un *pipeline* de entrega continua expone múltiples vectores de ataque. Cada componente del sistema tiene su propia forma de ser comprometido:

| Componente | Vector de ataque típico |
|---|---|
| Repositorio de código | *Push* directo a rama protegida; *PR* con código malicioso desde *fork* |
| Archivos de *workflow* | Modificación de `.github/workflows/` para exfiltrar secretos |
| *Actions* de terceros | Compromiso del código de la *Action*; *tag* o *branch* mutables |
| Dependencias del proyecto | CVE en paquete existente; paquete malicioso; *typosquatting* |
| Caché del *pipeline* | Envenenamiento de caché entre ramas o *workflows* |
| *Secrets* y credenciales | Filtración en *logs*; acceso desde *workflow* no autorizado |
| *Runners* | Escape del contenedor; artefactos persistentes entre *runs* |
| Registro de artefactos | Sustitución de imagen o paquete publicado |
| Sistema de *build* | Modificación del proceso de compilación antes de la firma |

Cada fila representa un punto donde un atacante puede introducir código malicioso o extraer credenciales sin que el software final muestre diferencias visibles en una revisión de código ordinaria.

## La amenaza no es teórica: incidentes documentados.

Cada vector de la tabla anterior corresponde a un incidente real. Los siguientes casos son los más relevantes para entender el contexto de este curso y por qué cada práctica del curso tiene una justificación técnica concreta frente a un ataque operativo conocido.

| Año | Incidente | Descripción |
|---|---|---|
| 2020 | [SolarWinds Orion](https://www.cisa.gov/news-events/cybersecurity-advisories/aa20-352a) | Sistema de build comprometido. Backdoor firmado distribuido a ~18.000 organizaciones. |
| 2021 | [Codecov bash uploader](https://about.codecov.io/apr-2021-post-mortem/) | Script de CI modificado. Secretos exfiltrados de miles de pipelines en producción. |
| 2021 | [ua-parser-js (npm)](https://www.cisa.gov/news-events/alerts/2021/10/22/malware-discovered-popular-npm-package-ua-parser-js) | Cuenta del mantenedor secuestrada. Malware en paquete con 8M+ descargas/semana. |
| 2021 | [Log4Shell (CVE-2021-44228)](https://nvd.nist.gov/vuln/detail/CVE-2021-44228) | CVE crítico en dependencia ubicua. Explotable en producción en millones de sistemas. |
| 2023 | [3CX supply chain](https://www.cisa.gov/news-events/alerts/2023/03/30/supply-chain-attack-against-3cxdesktopapp) | Dependencia comprometida que comprometió el instalador distribuido a clientes finales. |
| 2024 | [XZ Utils (CVE-2024-3094)](https://nvd.nist.gov/vuln/detail/CVE-2024-3094) | 2 años de ingeniería social. Backdoor en biblioteca de compresión crítica de Linux. |
| 2025 | [tj-actions/changed-files](https://www.cisa.gov/news-events/alerts/2025/03/18/supply-chain-compromise-third-party-tj-actionschanged-files-cve-2025-30066-and-reviewdogaction) | GitHub Action comprometida. Secretos impresos en logs de workflows de miles de repositorios. |

## SolarWinds (2020): comprometer el *build*.

El ataque a SolarWinds es el caso de referencia para entender por qué el sistema de construcción es infraestructura crítica.

Los atacantes (atribuidos al grupo APT29/Cozy Bear) comprometieron el proceso de *build* de Orion, el software de monitoreo de red de SolarWinds. Insertaron un *backdoor* (denominado SUNBURST) directamente en el proceso de compilación: el código fuente en el repositorio no contenía el *backdoor*, pero el binario compilado sí. El artefacto resultante fue firmado con los certificados legítimos de SolarWinds y distribuido como una actualización oficial.

El resultado fue que aproximadamente 18.000 organizaciones descargaron actualizaciones firmadas que contenían el *backdoor*. Entre los afectados: el Departamento del Tesoro de EE. UU., el Departamento de Seguridad Nacional, Microsoft, FireEye y decenas de empresas Fortune 500.

**Por qué este ataque importa para este curso:**

- La firma criptográfica del artefacto no es suficiente si el proceso que lo firmó está comprometido.
- El código auditado en el repositorio no garantiza que el binario distribuido sea su equivalente exacto.
- La respuesta técnica es *build* reproducible, *provenance* verificable y SBOM: poder demostrar que un artefacto específico fue construido desde un proceso conocido, en un *runner* verificable, desde un *commit* exacto, sin modificaciones en tránsito.

> **Referencias:** [CISA Advisory AA20-352A](https://www.cisa.gov/news-events/cybersecurity-advisories/aa20-352a) · [Mandiant — SUNBURST backdoor](https://cloud.google.com/blog/topics/threat-intelligence/evasive-attacker-leverages-solarwinds-supply-chain-compromises-with-sunburst-backdoor)

## Codecov (2021): exfiltrar secretos del pipeline.

Codecov es una herramienta de cobertura de código que se integra en pipelines de CI/CD mediante un script bash descargado desde sus servidores en tiempo de ejecución. En abril de 2021 se descubrió que ese script había sido modificado por atacantes que comprometieron el almacenamiento de artefactos de Codecov.

La modificación era mínima pero efectiva: una sola línea adicional que tomaba todas las variables de entorno del runner —donde viven los secretos del pipeline— y las enviaba a un servidor controlado por los atacantes. Cualquier pipeline que ejecutara el script durante el período de compromiso exfiltró sus credenciales sin ninguna señal visible de error.

Entre los afectados: Twilio, HashiCorp, Atlassian, Confluent y cientos de organizaciones adicionales. El período de exposición fue de aproximadamente dos meses antes del descubrimiento.

**Por qué este ataque importa:**

- Descargar y ejecutar scripts de URLs externas en el pipeline sin verificar su hash es una práctica de riesgo alto, independientemente de la reputación del proveedor.
- Los secretos en variables de entorno están disponibles para cualquier código que corra en el runner, sin importar si ese código es propio o de un tercero.
- La respuesta técnica es fijar todas las herramientas externas por hash y verificar la integridad antes de ejecutar. `uv sync --frozen` y la fijación de Actions por SHA siguen exactamente esta misma lógica.

> **Referencias:** [Codecov — Post-mortem abril 2021](https://about.codecov.io/apr-2021-post-mortem/) · [Rapid7 — análisis del compromiso](https://www.rapid7.com/blog/post/2021/04/16/codecov-discloses-supply-chain-compromise/)

## XZ Utils (2024): ingeniería social de largo plazo.

XZ Utils es una biblioteca de compresión incluida en prácticamente todas las distribuciones Linux. En febrero de 2024, el ingeniero de PostgreSQL Andres Freund descubrió por casualidad un backdoor en las versiones 5.6.0 y 5.6.1 de la biblioteca (CVE-2024-3094).

El ataque fue notable por su sofisticación y paciencia. Un contribuidor bajo el alias «Jia Tan» comenzó a aportar parches al proyecto en 2021. Durante dos años ganó confianza progresivamente, obtuvo acceso de commit, asumió el mantenimiento activo y presionó al mantenedor original usando identidades falsas que simulaban ser usuarios del proyecto insatisfechos con la velocidad de desarrollo. Finalmente, en 2024, insertó el backdoor en el código de inicialización de la biblioteca: un mecanismo que interceptaba la autenticación SSH en sistemas Linux con systemd.

El backdoor fue descubierto antes de llegar a las distribuciones estables por un margen muy estrecho.

**¿Por qué este ataque importa?**

- Las dependencias de código abierto son código de terceros que se ejecuta con los privilegios de la aplicación y del sistema operativo.
- El historial positivo de contribuciones de un mantenedor no garantiza que una versión específica sea segura.
- La respuesta técnica es fijar versiones por hash verificable (no solo número de versión), generar SBOM para cada artefacto construido, y escanear dependencias continuamente con herramientas como *Trivy*.

> **Referencias:** [NVD — CVE-2024-3094](https://nvd.nist.gov/vuln/detail/CVE-2024-3094) · [OpenSSF — análisis del backdoor](https://openssf.org/blog/2024/03/30/xz-backdoor-cve-2024-3094/)

## tj-actions/changed-files (2025): veneno en una Action popular.

`tj-actions/changed-files` es una GitHub Action usada en decenas de miles de repositorios para detectar qué archivos cambiaron en un pull request. En marzo de 2025 fue comprometida: el atacante modificó el código de la Action para que imprimiera los secretos del workflow en los logs de ejecución, visibles para cualquier persona con acceso de lectura al repositorio.

El vector fue el propio mecanismo de tags de GitHub: los tags son referencias mutables. Pueden ser movidos a un commit diferente en cualquier momento, sin aviso y sin dejar rastro obvio. Cualquier repositorio que referenciara la Action por tag en lugar de por SHA exacto del commit estaba expuesto.

```yaml
# referencia insegura: el tag puede moverse a cualquier commit
- uses: tj-actions/changed-files@v35

# referencia segura: el SHA de commit es inmutable
- uses: tj-actions/changed-files@<SHA-del-commit>
```

**Por qué este ataque importa para este curso:**

- Las Actions de terceros son código externo que se ejecuta con acceso completo a los secretos del workflow.
- Referenciar por tag es confiar en que nadie moverá ese tag a código malicioso: esa confianza no es verificable ni auditable.
- La respuesta técnica es fijar todas las Actions por SHA de commit inmutable y usar *Dependabot* para mantener esas referencias actualizadas automáticamente.

> **Referencias:** [CISA — CVE-2025-30066](https://www.cisa.gov/news-events/alerts/2025/03/18/supply-chain-compromise-third-party-tj-actionschanged-files-cve-2025-30066-and-reviewdogaction) · [Wiz — análisis técnico](https://www.wiz.io/blog/github-action-tj-actions-changed-files-supply-chain-attack-cve-2025-30066)

## Envenenamiento de caché en *GitHub Actions*.

El mecanismo de caché de *GitHub Actions* permite persistir directorios entre ejecuciones para acelerar *workflows*. La clave de caché puede construirse con el nombre de la rama, el *hash* del *lockfile* u otros datos del contexto.

El vector de ataque funciona así: un contribuidor con acceso a una rama de *feature* puede crear una entrada de caché con una clave que coincida con la que usará después la rama principal. Si un *workflow* posterior toma esa caché contaminada, puede estar instalando dependencias o ejecutando herramientas modificadas por el atacante, sin ninguna señal de error visible.

Este vector es relevante porque:

- la caché se comparte entre *workflows* del mismo repositorio por defecto;
- entradas creadas en ramas de *feature* pueden ser leídas por la rama principal;
- el contenido de la caché no tiene firma ni verificación de integridad por defecto.

**La respuesta técnica en este curso:** usar `uv sync --frozen` en el *pipeline* verifica los *hashes* SHA-256 de todas las dependencias contra `uv.lock` antes de instalar, independientemente de lo que contenga la caché. Si el contenido cacheado no coincide con los *hashes* del *lockfile*, la instalación falla con un error explícito antes de que cualquier código externo se ejecute.

> **Referencias:** [Adnan Khan — The Monsters in Your Build Cache](https://adnanthekhan.com/2024/05/06/the-monsters-in-your-build-cache-github-actions-cache-poisoning/)

## El patrón común detrás de los ataques.

Revisando los incidentes documentados, emerge un patrón consistente en cuatro elementos que se repiten en casi todos los casos:

**1. Confianza implícita en componentes externos.**
El *pipeline* descarga y ejecuta código —*scripts*, *Actions*, dependencias, herramientas de *build*— sin verificar que sea el mismo código que se usó en *builds* anteriores o que fue auditado por el equipo.

**2. Referencias mutables.**
*Tags* de *git*, URLs sin *hash*, versiones sin *lockfile*. El atacante no necesita acceso al repositorio propio; solo necesita modificar el recurso externo al que el *pipeline* hace referencia. El ataque ocurre antes de que cualquier revisión de *PR* sea relevante.

**3. Acceso a credenciales sin restricción de contexto.**
Los secretos están disponibles para cualquier código que corra en el *runner*, sin importar si ese código es propio, de un tercero o fue introducido mediante envenenamiento de caché o de una dependencia.

**4. Sin evidencia verificable del *build*.**
No existe forma de demostrar que el artefacto distribuido fue producido por el proceso esperado, desde el código fuente esperado, sin modificaciones en tránsito.

Este curso trabaja sobre exactamente estos cuatro problemas: fijar referencias por *hash* inmutable, verificar integridad en la instalación, acotar el acceso a credenciales por entorno y *job*, y generar evidencia verificable y firmada del proceso de construcción.

## El modelo de confianza de un *pipeline*.

Un *pipeline* *DevSecOps* hace explícito aquello en lo que confía y verifica esa confianza en lugar de asumirla. La tabla contrasta la postura de confianza implícita —común antes de los incidentes descritos— con la postura de confianza verificada que se propone:

| Componente | Confianza implícita (insegura) | Confianza verificada (este curso) |
|---|---|---|
| *Actions* de terceros | *Tag* mutable: `action/foo@v2` | SHA inmutable: `action/foo@<sha>` |
| Dependencias *Python* | `pip install` sin *hash* verificado | `uv sync --frozen` con *hashes* SHA-256 |
| Imagen base | `FROM python:latest` | `FROM python:3.12-slim@sha256:<hash>` |
| Credenciales *cloud* | JSON *key* de larga duración en *Secret* | Token OIDC efímero, válido por minutos |
| Origen del artefacto | Sin registro ni trazabilidad | SBOM + *provenance* (procedencia) firmada |
| Integridad del artefacto | Sin verificación post-*build* | Firma *Cosign* + *attestation* verificable |
| Secretos del *pipeline* | Disponibles para todo el *workflow* | Acotados por entorno y *job* específico |

El objetivo no es eliminar toda la confianza externa —eso haría el *pipeline* imposible de mantener—, sino hacerla **explícita, verificable y acotada al mínimo necesario**.

## SLSA: niveles de garantía de la cadena de suministro.

[*Supply-chain Levels for Software Artifacts*](https://slsa.dev/) (SLSA, pronunciado «salsa») es un marco desarrollado originalmente por Google y formalizado como estándar abierto. Define niveles progresivos de garantía sobre el proceso de construcción de software.

Los niveles no describen qué tan seguro es el *código*, sino qué tan verificable y resistente a manipulación es el *proceso que lo produjo*:

| Nivel | Garantía | Qué se puede demostrar |
|---|---|---|
| SLSA 0 | Ninguna | El artefacto existe |
| SLSA 1 | Documentación | El build está documentado; hay provenance (procedencia) básica |
| SLSA 2 | Resistencia básica | Build en servicio gestionado; provenance (procedencia) firmada y verificable |
| SLSA 3 | Aislamiento | Builder verificable independientemente; código fuente fijado; build hermético |

La importancia de SLSA en este contexto es concreta: GitHub Actions con `slsa-framework/slsa-github-generator` permite alcanzar SLSA 3 para artefactos publicados sin infraestructura adicional. Eso convierte la provenance (procedencia) de un concepto abstracto a una propiedad técnica verificable de cada imagen o paquete publicado.

La respuesta técnica a la pregunta «¿cómo sé que este artefacto es el que mi pipeline construyó?» es exactamente SLSA + firma + attestation, y conecta directamente con los vectores de SolarWinds y XZ Utils descritos antes.

## *Shift left*: mover los controles hacia el origen.

El término *shift left* viene de representar el ciclo de desarrollo como una línea de tiempo horizontal. Históricamente, las verificaciones de seguridad y calidad se hacían al final del proceso: auditorías antes del release, pruebas de penetración sobre el sistema en producción, revisiones de compliance en el sprint final.

*Shift left* invierte esa lógica: mover los controles hacia el inicio del proceso, donde detectar un problema es más rápido, más barato y más efectivo.

```
<-- más temprano ------------------------------------------- más tardío -->

Desarrollador   Pre-commit    PR / CI      QA / Staging    Producción
     |               |            |              |               |
  lint en IDE     ruff check   tests auto    pen test       incidente
  mypy local      secretos     SCA Trivy     compliance     CVE en prod
  ruff check      YAML lint    SBOM          auditoría      rollback
```

La razón por la que el desplazamiento tiene valor operativo es el costo asimétrico de corregir un problema:

- en el IDE: minutos, sin impacto en producción;
- en el PR/CI: horas, bloqueado antes del merge;
- en producción: días o semanas de incident response, comunicación a clientes, análisis forense, reputación dañada.

Shift left no significa que las verificaciones de etapas tardías son innecesarias. Significa que cada capa detecta lo que la anterior no pudo, y que la primera línea de defensa debe estar lo más cerca posible del origen del problema.

| Fase | Costo relativo | Controles típicos |
|---|---|---|
| IDE / desarrollo | 1× | lint, tipado estático, *mypy*, *ruff* |
| Pre-commit | 2× | hooks: *ruff*, secretos, yaml, conflictos no resueltos |
| Pull request / CI | 10× | SCA, SAST, tests automatizados, SBOM, escaneo de imagen |
| QA / staging | 40× | integración, regresión, validación funcional, compliance |
| Producción | 100× | incident response, rollback, comunicación, análisis forense |

## *Shift left* en las tres capas de este curso.

En este curso, *shift left* se implementa como un sistema de tres capas complementarias. Si una capa falla, la siguiente atrapa lo que la anterior no detectó.

**Capa local — máquina del desarrollador**

*pre-commit* ejecuta verificaciones antes de que el *commit* quede registrado: formato de código, sintaxis de *YAML* y JSON, detección de claves privadas accidentales, marcadores de conflicto no resueltos. Esta capa es la más rápida: falla en segundos, sin costo de red, sin esperar *runners*.

**Capa del repositorio — *pull request* y rama principal**

*GitHub Actions* ejecuta el conjunto completo de *quality gates*: *lint* con *ruff*, tipado con *mypy*, pruebas con *pytest*, análisis de composición de software con *Trivy*, *linting* del *Dockerfile* con *Hadolint*, inspección de imagen construida con *Dockle*. El *merge* queda bloqueado si cualquier *gate* falla. Ningún cambio defectuoso llega a la rama principal.

**Capa del artefacto — *build* y despliegue**

La imagen construida se escanea antes de publicarse. Se genera una SBOM. El artefacto se firma con *Cosign*. La firma y la *provenance* se adjuntan al registro de imágenes. Antes del despliegue, se verifica que el artefacto tiene *attestation* válida generada por el *pipeline* esperado. Esta capa garantiza que lo que llega a producción puede justificarse con evidencia técnica verificable.

La combinación de las tres capas es lo que hace al sistema **defendible**: no depende de que ninguna capa individual sea perfecta, sino de que las capas se refuercen mutuamente y cada una asuma que las anteriores pueden fallar.

## Este curso frente a las amenazas actuales.

El siguiente mapa relaciona cada amenaza documentada en este cuaderno con las prácticas concretas del curso que la contrarrestan:

| Amenaza | Práctica en este curso |
|---|---|
| Actions de terceros comprometidas | Fijar por SHA inmutable + *Dependabot* para Actions |
| Dependencias con CVE conocido | `uv.lock` con hashes SHA-256; *Trivy* SCA en CI |
| Envenenamiento de caché del pipeline | `uv sync --frozen`; verificación de hashes en instalación |
| Secretos exfiltrados del runner | OIDC efímero; secretos acotados por entorno y job |
| Artefacto modificado en tránsito | SBOM + firma *Cosign* + attestation SLSA verificable |
| Build no reproducible | Lockfiles; imagen base por digest; provenance SLSA |
| Credenciales estáticas de larga duración | Federación OIDC con GCP; sin JSON keys en secretos |
| Privilegio excesivo del pipeline | IAM por rol específico; service accounts acotados |
| Infraestructura creada manualmente | *Terraform* con estado remoto; IaC auditada en CI/CD |
| Sin visibilidad post-despliegue | Smoke tests; logs estructurados; criterio de rollback |

## Cierre.

La cadena de entrega continua es infraestructura crítica. Los incidentes documentados en este cuaderno no son excepciones ni eventos históricos cerrados: son el resultado de un desplazamiento sostenido del foco de ataque hacia el proceso de construcción y distribución del software, y los mismos vectores siguen siendo explotados con variaciones.

Este curso no responde a estos riesgos con alarma, sino con ingeniería: cada práctica del curso tiene una justificación técnica directa frente a un vector de ataque documentado y operativo.

Al terminar el curso, el objetivo no es solo saber usar *GitHub Actions*, *Terraform* o *Cosign*. Es entender por qué cada decisión técnica en el *pipeline* tiene consecuencias de seguridad, y cómo construir un sistema de entrega que sea rápido, auditable y defendible ante los vectores que hoy son operativos en producción.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>